In [2]:
# ============================================================
# STUDYMATE AI — INTELLIGENT STUDY ASSISTANT
# Complete Google Colab Project in ONE cell
# Modern dark UI + PDF upload/indexing + semantic search
# ============================================================

import os
import sys
import re
import subprocess
import shutil
from pathlib import Path

print("Installing required libraries...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "sentence-transformers", "pypdf", "gradio", "numpy"
])

import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# 1. PROJECT FOLDERS AND SETTINGS
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/studymate_ai")
KNOWLEDGE_DIR = PROJECT_DIR / "knowledge"
OUTPUT_DIR = PROJECT_DIR / "outputs"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.25

print("Project folder:", PROJECT_DIR)
print("Loading embedding model. The first download may take a few minutes...")
model = SentenceTransformer(MODEL_NAME)

CHUNKS = []
EMBEDDINGS = np.empty((0, 384), dtype=np.float32)

# ------------------------------------------------------------
# 2. PDF TEXT EXTRACTION AND CHUNKING
# ------------------------------------------------------------
def split_into_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r"\s+", " ", text or "").strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()
        if len(piece) >= 40:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks


def process_pdfs():
    documents = []
    errors = []

    pdf_paths = sorted(KNOWLEDGE_DIR.glob("*.pdf"))
    if not pdf_paths:
        errors.append("No PDF files are currently saved in the knowledge folder.")

    for pdf_path in pdf_paths:
        try:
            reader = PdfReader(str(pdf_path))
            if reader.is_encrypted:
                errors.append(f"{pdf_path.name}: encrypted PDF; skipped.")
                continue

            readable_pages = 0
            for page_num, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                if text.strip():
                    readable_pages += 1
                for chunk in split_into_chunks(text):
                    documents.append({
                        "text": chunk,
                        "source": pdf_path.name,
                        "page": page_num
                    })

            if readable_pages == 0:
                errors.append(
                    f"{pdf_path.name}: no selectable text found. "
                    "If this is a scanned PDF, OCR is needed."
                )
        except Exception as exc:
            errors.append(f"{pdf_path.name}: {exc}")

    return documents, errors


def rebuild_index():
    global CHUNKS, EMBEDDINGS
    documents, errors = process_pdfs()
    if not documents:
        CHUNKS = []
        EMBEDDINGS = np.empty((0, 384), dtype=np.float32)
        return 0, errors

    try:
        vectors = model.encode(
            [item["text"] for item in documents],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        CHUNKS = documents
        EMBEDDINGS = np.asarray(vectors, dtype=np.float32)
        return len(CHUNKS), errors
    except Exception as exc:
        CHUNKS = []
        EMBEDDINGS = np.empty((0, 384), dtype=np.float32)
        errors.append(f"Could not create embeddings: {exc}")
        return 0, errors


# Index PDFs that already exist from a previous run in this runtime.
initial_count, initial_errors = rebuild_index()
print("Initial text chunks indexed:", initial_count)
for warning in initial_errors:
    print("Note:", warning)

# ------------------------------------------------------------
# 3. SEMANTIC SEARCH
# ------------------------------------------------------------
def search_notes(question, top_k=5):
    if not question or not question.strip() or not CHUNKS:
        return []

    query_vector = model.encode(
        [question.strip()],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]
    scores = EMBEDDINGS @ query_vector
    indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for index in indices:
        item = dict(CHUNKS[int(index)])
        item["score"] = float(scores[int(index)])
        results.append(item)
    return results


def format_sources(results):
    if not results:
        return "No matching passages found."

    output = []
    for i, item in enumerate(results, start=1):
        output.append(
            f"### Passage {i}\n"
            f"**Source:** {item['source']}  \n"
            f"**Page:** {item['page']}  \n"
            f"**Similarity score:** {item['score']:.3f}\n\n"
            f"{item['text']}"
        )
    return "\n\n---\n\n".join(output)


# ------------------------------------------------------------
# 4. EXTRACTIVE TOPIC SUMMARY
# ------------------------------------------------------------
def create_summary(results, limit=5):
    sentences = []
    seen = set()

    for item in results:
        for sentence in re.split(r"(?<=[.!?])\s+", item["text"]):
            sentence = sentence.strip()
            if len(sentence.split()) < 6:
                continue
            key = sentence.lower()
            if key not in seen:
                seen.add(key)
                sentences.append((sentence, item["source"], item["page"]))

    if not sentences:
        return "Not enough readable text to summarize. Try another topic."

    output = ["## Summary from your study notes"]
    for i, (sentence, source, page) in enumerate(sentences[:limit], start=1):
        output.append(
            f"{i}. {sentence}\n\n**Source:** {source}, page {page}"
        )
    output.append(
        "\n*This is an extractive summary: it selects sentences from your PDF "
        "rather than generating new sentences with an LLM.*"
    )
    return "\n\n".join(output)


# ------------------------------------------------------------
# 5. BASIC PRACTICE MCQ GENERATOR
# ------------------------------------------------------------
STOP_WORDS = {
    "about", "after", "again", "also", "because", "before", "being",
    "between", "could", "does", "during", "each", "from", "have",
    "into", "more", "most", "other", "should", "some", "such", "than",
    "that", "their", "them", "then", "there", "these", "they", "this",
    "those", "through", "under", "using", "very", "what", "when",
    "where", "which", "while", "with", "would", "your", "will", "were",
    "been", "explain", "describe", "define", "write", "notes", "topic"
}


def create_mcqs(results, limit=5):
    sentences = []
    vocabulary = []

    for item in results:
        for sentence in re.split(r"(?<=[.!?])\s+", item["text"]):
            sentence = sentence.strip()
            if len(sentence.split()) >= 7:
                sentences.append({
                    "text": sentence,
                    "source": item["source"],
                    "page": item["page"]
                })

        words = re.findall(r"\b[A-Za-z][A-Za-z0-9+-]{4,}\b", item["text"])
        for word in words:
            if word.lower() not in STOP_WORDS and all(
                word.lower() != v.lower() for v in vocabulary
            ):
                vocabulary.append(word)

    output = [
        "## Practice Questions",
        "*Basic automatically generated fill-in-the-blank exercises. "
        "Check each answer against your notes.*"
    ]
    made = 0

    for item in sentences:
        if made >= limit:
            break

        words = re.findall(r"\b[A-Za-z][A-Za-z0-9+-]{4,}\b", item["text"])
        words = [w for w in words if w.lower() not in STOP_WORDS]
        if not words:
            continue

        answer = max(words, key=len)
        if len(answer) > 30:
            continue

        distractors = []
        for word in vocabulary:
            if word.lower() == answer.lower():
                continue
            if word.lower() in item["text"].lower():
                continue
            if all(word.lower() != d.lower() for d in distractors):
                distractors.append(word)
            if len(distractors) == 3:
                break

        if len(distractors) < 3:
            continue

        options = sorted([answer] + distractors, key=str.lower)
        labels = ["A", "B", "C", "D"]
        correct = labels[options.index(answer)]
        blank_sentence = re.sub(
            r"\b" + re.escape(answer) + r"\b",
            "________",
            item["text"],
            count=1,
            flags=re.IGNORECASE
        )

        output.append(
            f"**Q{made + 1}.** {blank_sentence}\n\n"
            + "\n".join(
                f"- {label}. {option}"
                for label, option in zip(labels, options)
            )
            + f"\n\n**Answer:** {correct}. {answer}\n\n"
            + f"**Source:** {item['source']}, page {item['page']}"
        )
        made += 1

    if made == 0:
        return (
            "Unable to create enough practice questions from these passages. "
            "Try a longer PDF with more readable text."
        )
    return "\n\n---\n\n".join(output)


# ------------------------------------------------------------
# 6. STUDY ASSISTANT CONTROLLER
# ------------------------------------------------------------
def study_mate(mode, question):
    if not question or not question.strip():
        return "⚠️ Please enter a question or topic."

    if not CHUNKS or EMBEDDINGS.shape[0] == 0:
        return (
            "## No study notes indexed yet\n\n"
            "1. Upload one or more PDF files above.\n"
            "2. Click **Process Documents**.\n"
            "3. Wait for the green success message, then try again."
        )

    results = search_notes(question, top_k=5)
    if not results:
        return "No matching passages found. Try another question."

    if results[0]["score"] < SIMILARITY_THRESHOLD:
        return (
            "# Not enough matching evidence\n\n"
            "I couldn't find strongly related passages in your study PDFs. "
            "Try different keywords or check the closest passages below.\n\n"
            + format_sources(results[:2])
        )

    if mode == "Ask a question":
        return (
            "# Relevant information from your notes\n\n"
            "These passages were retrieved from your PDFs using semantic "
            "similarity. This version shows source evidence; it does not yet "
            "use a generative LLM to write a new answer.\n\n"
            + format_sources(results[:3])
        )
    if mode == "Summarize a topic":
        return create_summary(results)
    if mode == "Generate practice MCQs":
        return create_mcqs(results)

    return "Please select a valid study mode."


# ------------------------------------------------------------
# 7. UPLOAD PDFS AND BUILD THE INDEX
# ------------------------------------------------------------
def upload_and_index(uploaded_files):
    if not uploaded_files:
        return (
            "⚠️ **No files selected.** Choose at least one PDF, "
            "then click Process Documents."
        )

    saved = []
    for file_path in uploaded_files:
        if not file_path:
            continue
        source = Path(file_path)
        if source.suffix.lower() != ".pdf":
            continue
        destination = KNOWLEDGE_DIR / source.name
        try:
            if source.resolve() != destination.resolve():
                shutil.copy2(source, destination)
            saved.append(source.name)
        except Exception as exc:
            return f"❌ Could not save {source.name}: {exc}"

    if not saved:
        return "⚠️ No valid PDF files selected. Please select .pdf files."

    try:
        total, index_errors = rebuild_index()
    except Exception as exc:
        return f"❌ Failed to build the search index: {exc}"

    if total == 0:
        message = (
            "## ⚠️ No readable study text was indexed\n\n"
            "The PDF may contain scanned images instead of selectable text, "
            "or it may be encrypted/invalid. Try a text-based PDF."
        )
        if index_errors:
            message += "\n\n### Details\n" + "\n".join(
                f"- {error}" for error in index_errors
            )
        return message

    sources = sorted({item["source"] for item in CHUNKS})
    message = (
        "# ✅ Study materials processed successfully!\n\n"
        f"- **PDF files indexed:** {len(sources)}\n"
        f"- **Text chunks indexed:** {total}\n\n"
        "### Files in your knowledge base\n"
        + "\n".join(f"- {name}" for name in sources)
        + "\n\nYou can now enter a topic below and click **Study Now**."
    )
    if index_errors:
        message += "\n\n### Notes\n" + "\n".join(
            f"- {error}" for error in index_errors
        )
    return message


# ------------------------------------------------------------
# 8. MODERN CSS
# ------------------------------------------------------------
CUSTOM_CSS = """
:root {
    --sm-bg: #0b1020;
    --sm-panel: #151d35;
    --sm-panel2: #1b2645;
    --sm-purple: #8b5cf6;
    --sm-blue: #3b82f6;
    --sm-text: #f8fafc;
    --sm-muted: #c4cce0;
}
.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
    background:
      radial-gradient(circle at 8% 0%, rgba(124,58,237,.18), transparent 30%),
      radial-gradient(circle at 100% 10%, rgba(37,99,235,.18), transparent 28%),
      var(--sm-bg) !important;
    color: var(--sm-text) !important;
    font-family: Inter, "Segoe UI", sans-serif !important;
    padding: 24px !important;
}
#sm-hero {
    padding: 30px 30px 26px !important;
    margin-bottom: 20px !important;
    border: 1px solid rgba(167,139,250,.35) !important;
    border-radius: 22px !important;
    background: linear-gradient(125deg, rgba(124,58,237,.25), rgba(37,99,235,.14), rgba(21,29,53,.96)) !important;
    box-shadow: 0 18px 55px rgba(0,0,0,.22) !important;
}
#sm-hero h1 {
    color: white !important;
    font-size: clamp(30px, 4vw, 44px) !important;
    line-height: 1.15 !important;
    letter-spacing: -1.2px !important;
    margin-bottom: 8px !important;
}
#sm-hero p { color: #dbeafe !important; font-size: 16px !important; }
h2, h3 { color: #e9ddff !important; font-weight: 750 !important; }
p, label, .prose { color: var(--sm-muted) !important; }
.block, .panel, fieldset, .form {
    background: rgba(21,29,53,.82) !important;
    border: 1px solid rgba(148,163,184,.17) !important;
    border-radius: 16px !important;
}
button {
    border-radius: 12px !important;
    font-weight: 650 !important;
    transition: transform .18s ease, box-shadow .18s ease !important;
}
button:hover { transform: translateY(-1px); }
button.primary, .primary {
    background: linear-gradient(100deg, #7c3aed, #2563eb) !important;
    color: #fff !important;
    border: 0 !important;
    box-shadow: 0 7px 22px rgba(99,102,241,.24) !important;
}
button.primary:hover, .primary:hover {
    box-shadow: 0 9px 26px rgba(124,58,237,.4) !important;
}
button[role="tab"] {
    border-radius: 10px !important;
    font-weight: 650 !important;
}
button[role="tab"][aria-selected="true"] {
    background: linear-gradient(100deg, #7c3aed, #2563eb) !important;
    color: white !important;
}
textarea, input, .wrap textarea, .wrap input {
    background: #0e172b !important;
    color: #f8fafc !important;
    border: 1px solid #354568 !important;
    border-radius: 12px !important;
}
textarea:focus, input:focus {
    border-color: #a78bfa !important;
    box-shadow: 0 0 0 2px rgba(167,139,250,.18) !important;
}
input::placeholder, textarea::placeholder { color: #8795b4 !important; }
.prose { line-height: 1.75 !important; }
.prose h1, .prose h2, .prose h3 { color: #e9ddff !important; }
.prose code { color: #bfdbfe !important; background: #101a31 !important; }
hr { border-color: #34415f !important; opacity: .75 !important; }
footer { opacity: .7 !important; }
@media (max-width: 768px) {
    .gradio-container { padding: 12px !important; }
    #sm-hero { padding: 22px 18px !important; }
}
"""

# ------------------------------------------------------------
# 9. BUILD THE GRADIO INTERFACE
# ------------------------------------------------------------
with gr.Blocks(
    title="StudyMate AI | Intelligent Study Assistant",
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=CUSTOM_CSS
) as demo:

    gr.Markdown(
        """
        # 📚 StudyMate AI
        ### Your personal study companion
        Upload your study PDFs, find relevant information, summarize topics,
        and practise with questions generated from your notes.
        """,
        elem_id="sm-hero"
    )

    with gr.Tabs():
        with gr.Tab("📖 Study Assistant"):
            gr.Markdown("## 1 · Build your study library")
            gr.Markdown(
                "Choose text-based PDF notes. After uploading, press "
                "**Process Documents** and wait for the success message."
            )

            pdf_upload = gr.File(
                label="📎 Upload PDF study materials",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )
            upload_button = gr.Button(
                "⚡ Process Documents",
                variant="primary",
                size="lg"
            )
            status = gr.Markdown(
                "📌 **Ready when you are.** Upload PDF files and click "
                "**Process Documents** to create your searchable study library."
            )
            upload_button.click(
                fn=upload_and_index,
                inputs=pdf_upload,
                outputs=status
            )

            gr.Markdown("---")
            gr.Markdown("## 2 · Ask, revise and practise")

            mode = gr.Radio(
                choices=[
                    "Ask a question",
                    "Summarize a topic",
                    "Generate practice MCQs"
                ],
                value="Ask a question",
                label="Choose your study mode"
            )
            question = gr.Textbox(
                label="What do you want to learn?",
                placeholder="Example: Explain methods in Java with examples",
                lines=3
            )
            ask_button = gr.Button(
                "✨ Study Now",
                variant="primary",
                size="lg"
            )
            answer = gr.Markdown(
                "Your study results will appear here after you process a PDF "
                "and submit a question."
            )
            ask_button.click(
                fn=study_mate,
                inputs=[mode, question],
                outputs=answer
            )
            question.submit(
                fn=study_mate,
                inputs=[mode, question],
                outputs=answer
            )

        with gr.Tab("🧭 How to Use"):
            gr.Markdown(
                """
                ## Get started in 3 steps

                **01 — Upload:** Select one or more PDF study notes.

                **02 — Process:** Click **Process Documents** and wait for
                the success message showing indexed files and text chunks.

                **03 — Learn:** Choose a study mode, enter a topic, and click
                **Study Now**.

                ### Available modes
                - **Ask a question:** Retrieves relevant passages from your PDFs.
                - **Summarize a topic:** Selects relevant sentences from your PDFs.
                - **Generate practice MCQs:** Creates basic fill-in-the-blank questions.

                ### Important notes
                - This prototype uses semantic retrieval, extractive summaries,
                  and rule-based practice questions. It does not yet use a
                  generative LLM to write full conversational answers.
                - Scanned/image-only PDFs may require OCR.
                - Verify important information against your original notes.
                - Files in this Colab runtime may be lost when the runtime resets.
                """
            )

    gr.Markdown(
        "StudyMate AI · Learn smarter, one document at a time",
        elem_id="sm-footer"
    )

# ------------------------------------------------------------
# 10. LAUNCH
# ------------------------------------------------------------
print("\n" + "=" * 55)
print("STUDYMATE AI IS READY")
print("=" * 55)
print("Existing indexed chunks:", len(CHUNKS))
print("Project directory:", PROJECT_DIR)
print("Upload PDFs inside the web app and click Process Documents.")
demo.launch(share=True)


Installing required libraries...
Project folder: /content/studymate_ai
Loading embedding model. The first download may take a few minutes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initial text chunks indexed: 80
Note: ADS-UNIT-3.pdf: no selectable text found. If this is a scanned PDF, OCR is needed.

STUDYMATE AI IS READY
Existing indexed chunks: 80
Project directory: /content/studymate_ai
Upload PDFs inside the web app and click Process Documents.


/tmp/ipykernel_3003/1699433279.py:505: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b8f341b358732a8a13.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
